# 05 — Results Notebook: Multilayer Shock Propagation Analysis

This notebook consolidates the thesis results from the damped shock model and reframes them through a multilayer lens, with countries treated as layers and sectors as nodes within each layer.

The emphasis here is on results tables, layer-level interpretation, robustness checks, and export-ready outputs for the thesis and final presentation.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, os.path.abspath('..'))

from src.network.builder import build_matrices
from src.dynamics.compute_vj import compute_Vj
from src.dynamics.compute_phi import compute_phi
from src.dynamics.compute_damping import compute_damping
from src.simulation.simulator import ShockSimulator

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

print('Imports ready')

def filter_nodes(nodes, index):
    return [n for n in nodes if n in index]

def country_of(node):
    return node.split('_')[0]

def aggregate_country_matrix(matrix, labels):
    country_labels = pd.Index([country_of(n) for n in labels])
    countries = pd.Index(sorted(country_labels.unique()))
    out = pd.DataFrame(0.0, index=countries, columns=countries)
    for i, src_country in enumerate(country_labels):
        for j, dst_country in enumerate(country_labels):
            out.loc[dst_country, src_country] += float(matrix[i, j])
    return out

def country_layer_metrics(A_df, rl_series):
    countries = pd.Index(sorted({country_of(n) for n in A_df.index}))
    layer_rows = []
    for c in countries:
        nodes = [n for n in A_df.index if country_of(n) == c]
        available_nodes = [n for n in nodes if n in rl_series.index]
        if not available_nodes:
            continue
        within = A_df.loc[nodes, nodes].to_numpy().sum()
        outgoing = A_df.loc[:, nodes].to_numpy().sum()
        incoming = A_df.loc[nodes, :].to_numpy().sum()
        external = max(outgoing + incoming - 2 * within, 0.0)
        layer_rows.append({
            'country': c,
            'n_nodes': len(available_nodes),
            'within_weight': within,
            'incoming_weight': incoming,
            'outgoing_weight': outgoing,
            'external_weight': external,
            'external_share': external / (within + external) if (within + external) > 0 else np.nan,
            'mean_rl': rl_series.reindex(available_nodes).mean(),
            'max_rl': rl_series.reindex(available_nodes).max(),
        })
    result = pd.DataFrame(layer_rows)
    if result.empty:
        return result
    return result.sort_values(['mean_rl', 'external_share'], ascending=False)

def country_spillover_matrix(A_df, baseline_traj, shocked_traj):
    nodes = list(A_df.index)
    countries = pd.Index(sorted({country_of(n) for n in nodes}))
    node_to_idx = {node: idx for idx, node in enumerate(nodes)}
    node_to_country = {node: country_of(node) for node in nodes}
    flow = pd.DataFrame(0.0, index=countries, columns=countries)

    loss_traj = np.maximum(baseline_traj - shocked_traj, 0.0)
    A_vals = A_df.to_numpy()

    for t in range(1, loss_traj.shape[0]):
        prev_loss = loss_traj[t - 1]
        for src_country in countries:
            src_nodes = [node for node in nodes if node_to_country[node] == src_country]
            src_idx = [node_to_idx[node] for node in src_nodes]
            if not src_idx:
                continue
            src_loss = prev_loss[src_idx]
            if np.allclose(src_loss.sum(), 0.0):
                continue
            contrib_by_dest_node = A_vals[:, src_idx] @ src_loss
            for dest_country in countries:
                dest_idx = [node_to_idx[node] for node in nodes if node_to_country[node] == dest_country]
                if dest_idx:
                    flow.loc[dest_country, src_country] += float(contrib_by_dest_node[dest_idx].sum())
    return flow

print('Helpers ready')

Imports ready
Helpers ready


In [5]:
Z, F, X, A, B = build_matrices(2018)
V, IRR_node, input_dist = compute_Vj(A, X, Z=Z)
phi_node, phi_country = compute_phi(lpi_path='../data/raw/lpi.xlsx', nodes=A.index, year=2018)
d, D = compute_damping(V, phi_node)

sim = ShockSimulator(A=A, X=X, D=D, f=None, n_iter=20, phi=1.0)

SHOCK_SCENARIOS = [
    ('RUS_C19', 'single_node', 'Energy: RUS petroleum refining'),
    ('RUS', 'single_country', 'Energy: Full Russia collapse'),
    (filter_nodes(['RUS_C19','UKR_C19','UKR_A01','UKR_B06','RUS_B06'], A.index), 'multi_node', 'Energy: Ukraine war proxy'),
    (filter_nodes(['SAU_B06','RUS_B06','ROW_B06'], A.index), 'multi_node', 'Oil: OPEC-style extraction shock'),
    (filter_nodes(['USA_K','GBR_K','IRL_K'], A.index), 'multi_node', 'Financial: 2008-style US+UK+IRL finance'),
    (filter_nodes(['USA_A01','BRA_A01','ARG_A01','UKR_A01','AUS_A01'], A.index), 'multi_node', 'Food: Major agricultural exporters shock'),
    (filter_nodes(['CHN_C26','KOR_C26','TWN_C26'], A.index), 'multi_node', 'Tech: Semiconductor shock CHN+KOR+TWN'),
    (filter_nodes([n for n in A.index if any(n.endswith(s) for s in ['_H49','_H50','_H51','_H52','_I'])], A.index), 'multi_node', 'COVID: Global transport + hospitality shutdown'),
]

all_results = {}
for shock, mode, label in SHOCK_SCENARIOS:
    all_results[label] = sim.run(shock=shock, mode=mode, label=label)

summary = sim.summary_table(list(all_results.values()))
summary.to_csv('../outputs/results_summary_multilayer.csv', index=False)
print(summary.to_string(index=False))

print('Scenario analysis complete')

Active nodes after zero-output removal: 3928
Computing V_j: Upstream Systemic Exposure
Method: Option C — Global Export Concentration (IRR)

[1/3] Building input distribution matrix...
  Input distribution : (3928, 3928)
  Zero-input sectors : 0

[2/3] Computing IRR: global export concentration per sector...
  Computing export flows (cross-border only)...
  This may take a moment for 3928 nodes...
  Nodes with positive exports : 3918
  Non-exporting nodes         : 10
  Total export volume         : 12398232.52

  Global EXPORT concentration IRR by sector code:
  Top 10 most concentrated (irreplaceable export markets):
sector_code
B05         0.222951
C13T15      0.182886
B06         0.154113
B09         0.145945
C302T309    0.140728
C301        0.136797
J58T60      0.119016
C26         0.117303
H53         0.109292
C27         0.106891

  Bottom 10 least concentrated (substitutable export markets):
sector_code
H52       0.030460
C10T12    0.034768
H49       0.036281
F         0.045223

c:\Users\Aditya\anaconda3\envs\wion\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


Damped model ready: A(I-D) computed
  Mean diagonal of D : 0.3545

Shock   : Energy: RUS petroleum refining
Mode    : single_node | Nodes: 1
Baseline — nodes RL>10%: 360 | Max RL: 1.0000 (RUS_C19)
Damped  — nodes RL>10%: 57 | Max RL: 1.0000 (RUS_C19)
Delta   — mean absorption: 0.0467 | Max absorption: 0.4890 (RUS_D)
Damped* — common-ref RL>10%: 57 | Max RL: 1.0000 (RUS_C19)
Delta*  — common-ref mean absorption: 0.0467 | Max absorption: 0.4890 (RUS_D)

Shock   : Energy: Full Russia collapse
Mode    : single_country | Nodes: 49
Baseline — nodes RL>10%: 992 | Max RL: 1.0000 (RUS_A01)
Damped  — nodes RL>10%: 216 | Max RL: 1.0000 (RUS_A01)
Delta   — mean absorption: 0.0655 | Max absorption: 0.4885 (BLR_B07)
Damped* — common-ref RL>10%: 216 | Max RL: 1.0000 (RUS_A01)
Delta*  — common-ref mean absorption: 0.0655 | Max absorption: 0.4885 (BLR_B07)

Shock   : Energy: Ukraine war proxy
Mode    : multi_node | Nodes: 5
Baseline — nodes RL>10%: 462 | Max RL: 1.0000 (RUS_B06)
Damped  — nodes RL>10%:

In [ ]:
country_tables = []
country_spillover_tables = []

for label, r in all_results.items():
    shock_nodes = r['shock_nodes']
    base_rl = r['RL_final_base'].drop(index=shock_nodes, errors='ignore')
    damp_rl = r['RL_final_damp'].drop(index=shock_nodes, errors='ignore') if r['RL_final_damp'] is not None else None

    base_layers = country_layer_metrics(A, base_rl)
    base_layers['scenario'] = label
    base_layers['metric'] = 'baseline'
    country_tables.append(base_layers)

    if damp_rl is not None:
        damp_layers = country_layer_metrics(A, damp_rl)
        damp_layers['scenario'] = label
        damp_layers['metric'] = 'damped'
        damp_layers['absorption_gain'] = base_layers.set_index('country').loc[damp_layers['country'], 'mean_rl'].to_numpy() - damp_layers['mean_rl'].to_numpy()
        country_tables.append(damp_layers)

    base_flow = country_spillover_matrix(A, r['baseline'], r['shocked_base'])
    base_flow_long = base_flow.stack().reset_index()
    base_flow_long.columns = ['destination_country', 'source_country', 'flow']
    base_flow_long['scenario'] = label
    base_flow_long['metric'] = 'baseline'
    country_spillover_tables.append(base_flow_long)

    if r['baseline_damp'] is not None and r['shocked_damp'] is not None:
        damp_flow = country_spillover_matrix(A, r['baseline_damp'], r['shocked_damp'])
        damp_flow_long = damp_flow.stack().reset_index()
        damp_flow_long.columns = ['destination_country', 'source_country', 'flow']
        damp_flow_long['scenario'] = label
        damp_flow_long['metric'] = 'damped'
        country_spillover_tables.append(damp_flow_long)

        delta_flow_long = (base_flow - damp_flow).stack().reset_index()
        delta_flow_long.columns = ['destination_country', 'source_country', 'flow']
        delta_flow_long['scenario'] = label
        delta_flow_long['metric'] = 'delta'
        country_spillover_tables.append(delta_flow_long)

country_results = pd.concat(country_tables, ignore_index=True)
country_results.to_csv('../outputs/country_layer_results.csv', index=False)

spillover_results = pd.concat(country_spillover_tables, ignore_index=True)
spillover_results.to_csv('../outputs/country_layer_spillovers.csv', index=False)
spillover_results.to_csv('../outputs/country_layer_dependency_matrices.csv', index=False)

display(country_results.sort_values(['scenario', 'metric', 'mean_rl'], ascending=[True, True, False]).head(20))
display(spillover_results.sort_values(['scenario', 'metric', 'flow'], ascending=[True, True, False]).head(20))

KeyboardInterrupt: 

In [ ]:
# Sensitivity to damping intensity scaling
scales = [0.5, 1.0, 1.5]
scale_rows = []

for scale in scales:
    D_scaled = np.clip(D * scale, 0.0, 1.0)
    sim_scaled = ShockSimulator(A=A, X=X, D=D_scaled, f=None, n_iter=20, phi=1.0)
    for shock, mode, label in SHOCK_SCENARIOS:
        r = sim_scaled.run(shock=shock, mode=mode, label=label)
        shock_nodes = r['shock_nodes']
        if r['RL_final_damp'] is None:
            continue
        base_mean = r['RL_final_base'].drop(index=shock_nodes, errors='ignore').mean()
        damp_mean = r['RL_final_damp'].drop(index=shock_nodes, errors='ignore').mean()
        scale_rows.append({
            'scenario': label,
            'scale': scale,
            'base_mean_rl': base_mean,
            'damp_mean_rl': damp_mean,
            'delta_mean_rl': base_mean - damp_mean,
        })

scale_results = pd.DataFrame(scale_rows)
scale_results.to_csv('../outputs/damping_scale_sensitivity.csv', index=False)

scenario_strength = summary[['Scenario', 'Base Mean RL', 'Damp Mean RL', 'Mean Absorption']].copy() if 'Damp Mean RL' in summary.columns else summary[['Scenario', 'Base Mean RL']].copy()
scenario_strength.to_csv('../outputs/scenario_strength_summary.csv', index=False)

print('Sensitivity outputs saved')
display(scale_results.head())

---
## Thesis Outputs

This notebook exports the main tables used in the thesis results chapter and provides the multilayer summaries needed for the final narrative and presentation.